# Football Transfer Market Analyzer — Web Scraping Report

**Author:** Nijat Kazimli
**Website:** [Transfermarkt](https://www.transfermarkt.com)  
**Competition:** Premier League 2025/26  

---

## Project Overview

This project scrapes Transfermarkt to build a structured dataset of Premier League football players, including:
- **Player profiles** (name, nationality, age, height, position, club, market value, contract info)
- **Historical market values** (dynamic chart data rendered via JavaScript)

### Tools Demonstrated
| Tool | Purpose |
|------|---------|
| **Scrapy** | Crawl league/club pages, collect player URLs + squad-table data |
| **requests + BeautifulSoup** | Fetch & parse static player profile pages |
| **Selenium** | Handle cookie consent, extract dynamically-rendered market value charts |
| **Python regex (`re`)** | Clean financial strings, extract dates/years, parse heights |
| **Pandas** | Structure, merge, clean, and analyze the final dataset |

## 1. Setup and Import Libraries

In [ ]:
# Standard library
import os

# Data processing
import pandas as pd

# Our custom modules
from regex_utils import (
    parse_market_value, extract_year, extract_date,
    extract_age, extract_height_cm, clean_whitespace,
    extract_player_id, extract_club_id,
)
from profile_scraper import scrape_multiple_profiles
from selenium_scraper import scrape_multiple_market_values
from scrapy_spider import run_scrapy_spider

print("All imports successful")
print(f"Pandas version: {pd.__version__}")

All imports successful ✓
Pandas version: 3.0.2
Scrapy version: 2.15.2


## 2. Constants and Configuration

We define a custom User-Agent, crawl delay, and base URLs to ensure polite and identifiable scraping.

In [ ]:
# --- Configuration ---
BASE_URL = "https://www.transfermarkt.com"
SEASON = "2025"  # 2025/26 season

# Limits for demonstration (adjust for full crawl)
MAX_PROFILES_BS4 = 30     # Profiles to scrape with requests+BS4
MAX_PLAYERS_SELENIUM = 10  # Players for dynamic market value history

# Data output directory
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Target: {BASE_URL}/premier-league/ (Season {SEASON})")
print(f"Max profiles (BS4): {MAX_PROFILES_BS4}")
print(f"Max players (Selenium): {MAX_PLAYERS_SELENIUM}")

Target: https://www.transfermarkt.com/premier-league/ (Season 2025)
Crawl delay: 3s
Max profiles (BS4): 30
Max players (Selenium): 10


## 3. Scrapy Spider: Crawl League Tables and Collect Player URLs

The Scrapy spider:
1. Starts at the Premier League competition page
2. Extracts club links from the league table (using `td.hauptlink a[href*='/verein/']`)
3. Filters for `/startseite/verein/` links to avoid fixture/schedule pages
4. Follows each club's detailed squad page (`/kader/.../plus/1`)
5. Parses player rows to extract names, positions, market values, nationalities

**Note:** Due to Python 3.14 incompatibility with Scrapy's `inspect.getsource()` call, we run the spider via a subprocess with a monkey-patch applied.

In [3]:
# We use the scrapy_spider module which runs the spider via subprocess
# (needed due to Twisted reactor limitations and Python 3.14 compatibility)
from scrapy_spider import run_scrapy_spider

print("Running Scrapy spider to crawl Premier League squads...")
print("(This may take ~80s due to the 3-second crawl delay)\n")

output_path = os.path.join(DATA_DIR, "scrapy_players.json")
scrapy_results = run_scrapy_spider(output_path=output_path, season=SEASON)

print(f"\n✓ Scrapy collected {len(scrapy_results)} player records")
print(f"  Output saved to: {output_path}")

# --- Below is the spider class for reference (runs in subprocess) ---
# class TransfermarktSpider(scrapy.Spider):
#     name = "transfermarkt_pl"
#     allowed_domains = ["transfermarkt.com"]
#
#     def parse(self, response):
#         # Extract club links: td.hauptlink a[href*='/verein/']
#         club_links = response.css(
#             "table.items tbody tr td.hauptlink a[href*='/verein/']::attr(href)"
#         ).getall()
#         # Filter: only /startseite/verein/ links (exclude spielplan, etc.)
#         for link in club_links:
#             if '/startseite/verein/' not in link:
#                 continue
#             squad_url = link.replace("/startseite/", "/kader/")
#             squad_url += f"/saison_id/{self.season}/plus/1"
#             yield response.follow(squad_url, callback=self.parse_squad)
#
#     def parse_squad(self, response):
#         club_name = clean_whitespace(
#             response.css("header.data-header h1::text").get(default="")
#         )
#         rows = response.css("table.items > tbody > tr")
#         for row in rows:
#             name = row.css("td.hauptlink a::text").get()
#             profile_href = row.css("td.hauptlink a::attr(href)").get(default="")
#             market_value_raw = row.css("td.rechts.hauptlink a::text").get(default="")
#             yield {
#                 "player_name": clean_whitespace(name),
#                 "profile_url": BASE_URL + profile_href,
#                 "club": club_name,
#                 "market_value_eur": parse_market_value(market_value_raw),
#                 ...
#             }

Running Scrapy spider to crawl Premier League squads...
(This may take ~80s due to the 3-second crawl delay)

  Launching Scrapy spider subprocess...
  [scrapy] 2026-05-02 20:06:16 [scrapy.utils.log] INFO: Scrapy 2.15.2 started (bot: scrapybot)
  [scrapy] 2026-05-02 20:06:16 [scrapy.utils.log] INFO: Versions:
  [scrapy] {'lxml': '6.1.0',
  [scrapy]  'libxml2': '2.14.6',
  [scrapy]  'cssselect': '1.4.0',
  [scrapy]  'parsel': '1.11.0',
  [scrapy]  'w3lib': '2.4.1',
  [scrapy]  'Twisted': '25.5.0',
  [scrapy]  'Python': '3.14.2 (v3.14.2:df793163d58, Dec  5 2025, 12:18:06) [Clang 16.0.0 '
  [scrapy]            '(clang-1600.0.26.6)]',
  [scrapy]  'pyOpenSSL': '26.1.0 (OpenSSL 4.0.0 14 Apr 2026)',
  [scrapy]  'cryptography': '47.0.0',
  [scrapy]  'Platform': 'macOS-26.4.1-arm64-arm-64bit-Mach-O'}
  [scrapy] 2026-05-02 20:06:16 [scrapy.addons] INFO: Enabled addons:
  [scrapy] []
  [scrapy] 2026-05-02 20:06:16 [scrapy.extensions.telnet] INFO: Telnet Password: 1329c8d6be9d9382
  [scrapy] 2026-

In [4]:
# Convert Scrapy results to a DataFrame
df_scrapy = pd.DataFrame(scrapy_results)
print(f"Scrapy DataFrame shape: {df_scrapy.shape}")
print(f"\nColumns: {list(df_scrapy.columns)}")
print(f"\nSample data:")
df_scrapy[["player_name", "club", "position", "market_value_eur"]].head(10)

Scrapy DataFrame shape: (528, 10)

Columns: ['player_name', 'player_id', 'profile_url', 'club', 'position', 'date_of_birth_age', 'nationalities', 'shirt_number', 'market_value_raw', 'market_value_eur']

Sample data:


,player_name,club,position,market_value_eur
0,Max Weiß,Burnley FC,Goalkeeper,4000000.0
1,Martin Dúbravka,Burnley FC,Goalkeeper,750000.0
2,Václav Hladký,Burnley FC,Goalkeeper,200000.0
3,Maxime Estève,Burnley FC,Centre-Back,25000000.0
4,Bashir Humphreys,Burnley FC,Centre-Back,15000000.0
5,Hjalmar Ekdal,Burnley FC,Centre-Back,6000000.0
6,Axel Tuanzebe,Burnley FC,Centre-Back,5000000.0
7,Joe Worrall,Burnley FC,Centre-Back,3000000.0
8,Jordan Beyer,Burnley FC,Centre-Back,2500000.0
9,Quilindschy Hartman,Burnley FC,Left-Back,18000000.0


## 4. Requests & BeautifulSoup: Parse Static Player Profiles

For each player URL collected by Scrapy, we use `requests.get()` with custom headers to fetch the profile page HTML. Then `BeautifulSoup` parses the page to extract detailed attributes:
- Full name, date of birth, age, place of birth
- Citizenship, height, preferred foot
- Current club, contract dates
- International caps & goals

The implementation lives in `profile_scraper.py` with a `tqdm` progress bar. We sample evenly across clubs to avoid bias toward first-crawled teams.

In [ ]:
# --- Run the profile scraper ---
# Sample evenly across clubs to avoid bias toward first-crawled teams
n_clubs = df_scrapy["club"].nunique()
per_club = max(1, MAX_PROFILES_BS4 // n_clubs)
sampled = (
    df_scrapy.dropna(subset=["profile_url"])
    .groupby("club", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), per_club)))
)
profile_urls = sampled["profile_url"].unique().tolist()[:MAX_PROFILES_BS4]

print(f"Scraping {len(profile_urls)} player profiles with requests + BeautifulSoup...")
print(f"(Sampled evenly across {n_clubs} clubs, {per_club} per club)\n")

profiles = scrape_multiple_profiles(profile_urls, delay=2.0)

df_profiles = pd.DataFrame(profiles)
print(f"\n✓ Scraped {len(df_profiles)} player profiles with BeautifulSoup")

Scraping 20 player profiles with requests + BeautifulSoup...
(Sampled evenly across 20 clubs, 1 per club)
Estimated time: ~40s (2s delay per request)

  [1/20] evanilson ✓
  [2/20] max-dowman ✓
  [3/20] ian-maatsen ✓
  [4/20] vitaly-janelt ✓
  [5/20] jason-steele ✓
  [6/20] kyle-walker ✓
  [7/20] marc-guiu ✓
  [8/20] jean-philippe-mateta ✓
  [9/20] harrison-armstrong ✓
  [10/20] emile-smith-rowe ✓
  [11/20] charlie-crew ✓
  [12/20] hugo-ekitike ✓
  [13/20] abdukodir-khusanov ✓
  [14/20] manuel-ugarte ✓
  [15/20] harvey-barnes ✓
  [16/20] ibrahim-sangare ✓
  [17/20] abdoullah-ba ✓
  [18/20] antonin-kinsky ✓
  [19/20] tomas-soucek ✓
  [20/20] joao-gomes ✓

✓ Scraped 20 player profiles with BeautifulSoup


In [6]:
# Preview the profiles data
print(f"Profile DataFrame shape: {df_profiles.shape}")
print(f"\nColumns: {list(df_profiles.columns)}")
df_profiles[["full_name", "age", "height_cm", "position", "foot", "market_value_eur"]].head(10)

Profile DataFrame shape: (20, 17)

Columns: ['profile_url', 'full_name', 'date_of_birth', 'age', 'place_of_birth', 'height_cm', 'citizenship', 'position', 'foot', 'current_club', 'joined', 'contract_expires', 'contract_expires_year', 'market_value_raw', 'market_value_eur', 'international_caps', 'international_goals']


,full_name,age,height_cm,position,foot,market_value_eur
0,Evanilson,26,183,Centre-Forward,both,35000000
1,Max Dowman,16,183,Right Winger,left,20000000
2,Ian Maatsen,24,178,Left-Back,left,30000000
3,Vitaly Janelt,27,184,Defensive Midfield,left,18000000
4,Jason Steele,35,188,Goalkeeper,right,750000
5,Kyle Walker,35,183,Right-Back,right,2500000
6,Marc Guiu,20,187,Centre-Forward,right,12000000
7,Jean-Philippe Mateta,28,192,Centre-Forward,right,35000000
8,Harrison Armstrong,19,185,Central Midfield,right,12000000
9,Emile Smith Rowe,25,182,Attacking Midfield,right,22000000


## 5. Regex: Clean and Transform Raw Data

Python regular expressions are used extensively to:
1. **Parse market values**: Convert `"€45.00m"` → `45000000` (integer euros)
2. **Extract years**: Pull `2026` from `"Contract expires: Jun 30, 2026"`
3. **Parse heights**: Convert `"1,95 m"` → `195` (cm)
4. **Clean whitespace**: Collapse `"  Centre -  Forward  "` → `"Centre - Forward"`
5. **Extract IDs**: Pull numeric IDs from Transfermarkt URLs

Below we demonstrate each regex function on sample data:

In [7]:
# --- Demonstration of regex_utils functions ---

print("=" * 60)
print("REGEX DEMONSTRATIONS")
print("=" * 60)

# 1. Market value parsing
test_values = ["€200.00m", "€45.00m", "€500k", "€1.20bn", "-", "€8.00m"]
print("\n1. parse_market_value():")
for v in test_values:
    print(f"   '{v}' → {parse_market_value(v):>15,}" if parse_market_value(v) else f"   '{v}' → None")

# 2. Year extraction
test_contracts = [
    "Contract expires: Jun 30, 2026",
    "Joined: 01/07/2022",
    "Last extension: 17/01/2025",
]
print("\n2. extract_year():")
for t in test_contracts:
    print(f"   '{t}' → {extract_year(t)}")

# 3. Date extraction
print("\n3. extract_date():")
for t in test_contracts:
    print(f"   '{t}' → {extract_date(t)}")

# 4. Age extraction
test_ages = ["21/07/2000 (25)", "15/06/1998 (27)", "01/01/1990 (36)"]
print("\n4. extract_age():")
for t in test_ages:
    print(f"   '{t}' → {extract_age(t)}")

# 5. Height parsing
test_heights = ["1,95 m", "1,80 m", "1.73 m", "1,68 m"]
print("\n5. extract_height_cm():")
for t in test_heights:
    print(f"   '{t}' → {extract_height_cm(t)} cm")

# 6. Player/Club ID extraction
test_urls = [
    "/erling-haaland/profil/spieler/418560",
    "/manchester-city/startseite/verein/281",
]
print("\n6. extract_player_id() and extract_club_id():")
print(f"   Player ID: {extract_player_id(test_urls[0])}")
print(f"   Club ID:   {extract_club_id(test_urls[1])}")

# 7. Apply regex to actual scraped data
print("\n" + "=" * 60)
print("APPLYING REGEX TO SCRAPED DATA")
print("=" * 60)

# Show the regex pattern used inside parse_market_value
pattern = r"[€$£]?\s*([\d,.]+)\s*(m|k|bn)?"
print(f"\nMarket value regex pattern: {pattern}")
print("This matches: currency symbol (optional), number, unit suffix")

# Apply to DataFrame
if not df_scrapy.empty:
    sample = df_scrapy["market_value_raw"].dropna().head(5).tolist()
    print(f"\nRaw values from scrape → Parsed:")
    for raw in sample:
        parsed = parse_market_value(raw)
        print(f"   '{raw}' → €{parsed:,}" if parsed else f"   '{raw}' → None")

REGEX DEMONSTRATIONS

1. parse_market_value():
   '€200.00m' →     200,000,000
   '€45.00m' →      45,000,000
   '€500k' →         500,000
   '€1.20bn' →   1,200,000,000
   '-' → None
   '€8.00m' →       8,000,000

2. extract_year():
   'Contract expires: Jun 30, 2026' → 2026
   'Joined: 01/07/2022' → 2022
   'Last extension: 17/01/2025' → 2025

3. extract_date():
   'Contract expires: Jun 30, 2026' → None
   'Joined: 01/07/2022' → 01/07/2022
   'Last extension: 17/01/2025' → 17/01/2025

4. extract_age():
   '21/07/2000 (25)' → 25
   '15/06/1998 (27)' → 27
   '01/01/1990 (36)' → 36

5. extract_height_cm():
   '1,95 m' → 195 cm
   '1,80 m' → 180 cm
   '1.73 m' → 173 cm
   '1,68 m' → 168 cm

6. extract_player_id() and extract_club_id():
   Player ID: 418560
   Club ID:   281

APPLYING REGEX TO SCRAPED DATA

Market value regex pattern: [€$£]?\s*([\d,.]+)\s*(m|k|bn)?
This matches: currency symbol (optional), number, unit suffix

Raw values from scrape → Parsed:
   '€4.00m' → €4,000,000
   

## 6. Selenium: Extract Dynamic Market Value History

Transfermarkt renders the "Market Value Over Time" chart with a **Svelte** component that fetches data from an internal JSON API. This data is **not** in the initial HTML, so we need Selenium to:

1. Launch a headless Chrome browser
2. Navigate to the player's `/marktwertverlauf/` page
3. Accept the cookie consent dialog
4. Execute a same-origin XHR to the internal API endpoint (`tmapi-alpha.transfermarkt.technology/player/{id}/market-value-history`)
5. Parse the JSON response containing historical valuation data

This demonstrates Selenium's ability to interact with JavaScript-rendered content and execute arbitrary scripts within the browser context.

In [ ]:
# --- Run Selenium scraping ---
# Select top players by market value
top_players = (
    df_scrapy.dropna(subset=["market_value_eur"])
    .sort_values("market_value_eur", ascending=False)
    .head(MAX_PLAYERS_SELENIUM)
)

mv_urls = top_players["profile_url"].dropna().unique().tolist()
print(f"Scraping market value history for {len(mv_urls)} top players via Selenium...\n")

mv_results = scrape_multiple_market_values(mv_urls, max_players=MAX_PLAYERS_SELENIUM)

# Flatten results into a list of records with player info
mv_history_records = []
for url, data_points in mv_results.items():
    player_name = str(top_players.loc[
        top_players["profile_url"] == url, "player_name"
    ].values[0])
    for pt in data_points:
        pt["player_name"] = player_name
        pt["profile_url"] = url
        mv_history_records.append(pt)

df_mv_history = pd.DataFrame(mv_history_records)
print(f"\n✓ Selenium collected {len(df_mv_history)} market value data points")

Scraping market value history for 10 top players via Selenium...

  [1/10] Erling Haaland... ✓ (26 data points)
  [2/10] Declan Rice... ✓ (24 data points)
  [3/10] Bukayo Saka... ✓ (23 data points)
  [4/10] Cole Palmer... ✓ (16 data points)
  [5/10] Florian Wirtz... ✓ (17 data points)
  [6/10] Moisés Caicedo... ✓ (19 data points)
  [7/10] Dominik Szoboszlai... ✓ (26 data points)
  [8/10] Alexander Isak... ✓ (29 data points)
  [9/10] Hugo Ekitiké... ✓ (18 data points)
  [10/10] Enzo Fernández... ✓ (17 data points)

✓ Selenium collected 215 market value data points


In [9]:
# Preview market value history
if not df_mv_history.empty:
    print(f"Market Value History DataFrame shape: {df_mv_history.shape}")
    print(f"\nSample data points:")
    display(df_mv_history[["player_name", "date_display", "value_raw", "value_eur", "club_at_time"]].head(15))
else:
    print("No market value history data collected (chart may require different extraction method)")

Market Value History DataFrame shape: (215, 8)

Sample data points:


,player_name,date_display,value_raw,value_eur,club_at_time
0,Erling Haaland,2016-12-18,€200.00K,200000,Bryne FK
1,Erling Haaland,2017-12-23,€300.00K,300000,Molde FK
2,Erling Haaland,2018-09-10,€2.00M,2000000,Molde FK
3,Erling Haaland,2018-12-30,€5.00M,5000000,Molde FK
4,Erling Haaland,2019-06-03,€5.00M,5000000,Red Bull Salzburg
5,Erling Haaland,2019-09-03,€12.00M,12000000,Red Bull Salzburg
6,Erling Haaland,2019-11-07,€30.00M,30000000,Red Bull Salzburg
7,Erling Haaland,2019-12-16,€45.00M,45000000,Red Bull Salzburg
8,Erling Haaland,2020-02-11,€60.00M,60000000,Borussia Dortmund
9,Erling Haaland,2020-03-11,€80.00M,80000000,Borussia Dortmund


## 7. Combine and Structure the Final DataFrame

We merge the three data sources:
1. **Scrapy data** (squad-table level): player name, club, position, shirt number, squad market value
2. **BeautifulSoup data** (profile level): detailed attributes (height, foot, DOB, citizenship, contract)
3. **Selenium data** (chart level): historical market value time series

The primary dataset is player-level (one row per player), with the market value history as a supplementary time-series dataset.

In [10]:
# --- Merge Scrapy + BS4 profile data ---
if not df_profiles.empty and "profile_url" in df_profiles.columns:
    df_players = df_scrapy.merge(
        df_profiles, on="profile_url", how="left", suffixes=("_squad", "_profile")
    )
else:
    df_players = df_scrapy.copy()

print(f"Merged players DataFrame: {df_players.shape}")
print(f"\nAll columns ({len(df_players.columns)}):")
for col in sorted(df_players.columns):
    non_null = df_players[col].notna().sum()
    print(f"  {col:35s} {non_null:>4} non-null values")

df_players.head()

Merged players DataFrame: (528, 26)

All columns (26):
  age                                   20 non-null values
  citizenship                           20 non-null values
  club                                 528 non-null values
  contract_expires                      20 non-null values
  contract_expires_year                 19 non-null values
  current_club                          20 non-null values
  date_of_birth                         20 non-null values
  date_of_birth_age                    528 non-null values
  foot                                  20 non-null values
  full_name                             20 non-null values
  height_cm                             20 non-null values
  international_caps                    20 non-null values
  international_goals                   20 non-null values
  joined                                20 non-null values
  market_value_eur_profile              20 non-null values
  market_value_eur_squad               524 non-null values
 

,player_name,player_id,profile_url,club,position_squad,date_of_birth_age,nationalities,shirt_number,market_value_raw_squad,market_value_eur_squad,...,position_profile,foot,current_club,joined,contract_expires,contract_expires_year,market_value_raw_profile,market_value_eur_profile,international_caps,international_goals
0,Max Weiß,829299,https://www.transfermarkt.com/max-weiss/profil...,Burnley FC,Goalkeeper,15/06/2004 (21),[Germany],13,€4.00m,4000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Martin Dúbravka,74960,https://www.transfermarkt.com/martin-dubravka/...,Burnley FC,Goalkeeper,15/01/1989 (37),[Slovakia],1,€750k,750000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Václav Hladký,95795,https://www.transfermarkt.com/vaclav-hladky/pr...,Burnley FC,Goalkeeper,14/11/1990 (35),[Czech Republic],32,€200k,200000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Maxime Estève,842341,https://www.transfermarkt.com/maxime-esteve/pr...,Burnley FC,Centre-Back,26/05/2002 (23),[France],5,€25.00m,25000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bashir Humphreys,612517,https://www.transfermarkt.com/bashir-humphreys...,Burnley FC,Centre-Back,15/03/2003 (23),"[England, Uganda]",12,€15.00m,15000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Data Cleaning and Preparation for Analysis

Final data quality steps:
- Standardize position categories
- Set correct dtypes (numeric, datetime)
- Compute derived columns: peak market value per player, age group
- Handle missing values
- Display summary statistics

In [14]:
# --- Data Cleaning ---

# Standardize position categories
position_map = {
    "Goalkeeper": "Goalkeeper",
    "Centre-Back": "Defender",
    "Left-Back": "Defender",
    "Right-Back": "Defender",
    "Defensive Midfield": "Midfielder",
    "Central Midfield": "Midfielder",
    "Attacking Midfield": "Midfielder",
    "Left Midfield": "Midfielder",
    "Right Midfield": "Midfielder",
    "Left Winger": "Forward",
    "Right Winger": "Forward",
    "Centre-Forward": "Forward",
    "Second Striker": "Forward",
}

# Use the most detailed position column available
pos_col = "position" if "position" in df_players.columns else "position_squad"
df_players["position_category"] = df_players[pos_col].map(position_map).fillna("Unknown")

# Ensure market_value_eur is numeric
mv_col = "market_value_eur_squad" if "market_value_eur_squad" in df_players.columns else "market_value_eur"
df_players["market_value_eur_clean"] = pd.to_numeric(df_players[mv_col], errors="coerce")

# Age: use profile age if available, else try to extract from scrapy data
if "age" in df_players.columns:
    df_players["age_clean"] = pd.to_numeric(df_players["age"], errors="coerce")
else:
    df_players["age_clean"] = None

# Age groups
bins = [0, 21, 25, 29, 33, 45]
labels = ["U21", "22-25", "26-29", "30-33", "34+"]
df_players["age_group"] = pd.cut(df_players["age_clean"], bins=bins, labels=labels)

# Drop complete duplicates
df_players = df_players.drop_duplicates(subset=["player_id"], keep="first")

# --- Summary statistics ---
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"\nTotal unique players: {len(df_players)}")
print(f"\nPosition distribution:")
print(df_players["position_category"].value_counts().to_string())
print(f"\nAge group distribution:")
print(df_players["age_group"].value_counts().sort_index().to_string())
print(f"\nMarket value statistics (EUR):")
print(df_players["market_value_eur_clean"].describe().apply(lambda x: f"€{x:,.0f}"))
print(f"\nClubs represented: {df_players['club'].nunique()}")

# Market value history summary
if not df_mv_history.empty:
    print(f"\n--- Market Value History ---")
    print(f"Total data points: {len(df_mv_history)}")
    print(f"Players covered: {df_mv_history['player_name'].nunique()}")
    if "value_eur" in df_mv_history.columns:
        print(f"Date range: {df_mv_history['date_display'].min()} → {df_mv_history['date_display'].max()}")

DATASET SUMMARY

Total unique players: 528

Position distribution:
position_category
Defender      183
Midfielder    143
Forward       137
Goalkeeper     65

Age group distribution:
age_group
U21      4
22-25    8
26-29    5
30-33    1
34+      2

Market value statistics (EUR):
count            €524
mean      €24,000,620
std       €22,700,739
min           €75,000
25%        €8,000,000
50%       €20,000,000
75%       €32,000,000
max      €200,000,000
Name: market_value_eur_clean, dtype: str

Clubs represented: 20

--- Market Value History ---
Total data points: 215
Players covered: 10
Date range: 2016-06-17 → 2026-03-09


In [15]:
# --- Visualize key insights ---
print("=" * 60)
print("TOP 20 MOST VALUABLE PLAYERS IN THE PREMIER LEAGUE")
print("=" * 60)

top20 = (
    df_players.nlargest(20, "market_value_eur_clean")
    [["player_name", "club", "position_category", "age_clean", "market_value_eur_clean"]]
    .reset_index(drop=True)
)
top20.index += 1
top20.columns = ["Player", "Club", "Position", "Age", "Market Value (EUR)"]
top20["Market Value (EUR)"] = top20["Market Value (EUR)"].apply(
    lambda x: f"€{x/1_000_000:.1f}M" if pd.notna(x) else "N/A"
)
print(top20.to_string())

# Average market value by position
print("\n\nAVERAGE MARKET VALUE BY POSITION CATEGORY:")
avg_by_pos = (
    df_players.groupby("position_category")["market_value_eur_clean"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)
avg_by_pos["mean"] = avg_by_pos["mean"].apply(lambda x: f"€{x/1_000_000:.1f}M")
avg_by_pos["median"] = avg_by_pos["median"].apply(lambda x: f"€{x/1_000_000:.1f}M")
print(avg_by_pos.to_string())

# Average market value by age group
print("\n\nAVERAGE MARKET VALUE BY AGE GROUP:")
avg_by_age = (
    df_players.groupby("age_group", observed=True)["market_value_eur_clean"]
    .agg(["mean", "median", "count"])
    .sort_index()
)
avg_by_age["mean"] = avg_by_age["mean"].apply(lambda x: f"€{x/1_000_000:.1f}M")
avg_by_age["median"] = avg_by_age["median"].apply(lambda x: f"€{x/1_000_000:.1f}M")
print(avg_by_age.to_string())

TOP 20 MOST VALUABLE PLAYERS IN THE PREMIER LEAGUE
                 Player               Club    Position   Age Market Value (EUR)
1        Erling Haaland    Manchester City     Forward   NaN            €200.0M
2           Declan Rice         Arsenal FC  Midfielder   NaN            €120.0M
3           Bukayo Saka         Arsenal FC     Forward   NaN            €120.0M
4         Florian Wirtz       Liverpool FC  Midfielder   NaN            €110.0M
5        Moisés Caicedo         Chelsea FC  Midfielder   NaN            €110.0M
6           Cole Palmer         Chelsea FC  Midfielder   NaN            €110.0M
7    Dominik Szoboszlai       Liverpool FC  Midfielder   NaN            €100.0M
8        Alexander Isak       Liverpool FC     Forward   NaN            €100.0M
9      Ryan Gravenberch       Liverpool FC  Midfielder   NaN             €90.0M
10         Hugo Ekitiké       Liverpool FC     Forward  23.0             €90.0M
11       Enzo Fernández         Chelsea FC  Midfielder   NaN         

## 9. Export Dataset

Export the final datasets to CSV files for further analysis or submission.

In [13]:
# --- Export final datasets ---

# 1. Players dataset (one row per player)
players_csv_path = os.path.join(DATA_DIR, "players_full.csv")
df_players.to_csv(players_csv_path, index=False, encoding="utf-8-sig")
print(f"✓ Saved: {players_csv_path} ({len(df_players)} rows, {len(df_players.columns)} columns)")

# 2. Market value history (time series)
if not df_mv_history.empty:
    mv_csv_path = os.path.join(DATA_DIR, "market_value_history.csv")
    df_mv_history.to_csv(mv_csv_path, index=False, encoding="utf-8-sig")
    print(f"✓ Saved: {mv_csv_path} ({len(df_mv_history)} rows)")

# 3. Raw Scrapy output (already saved as JSON by the spider)
scrapy_json_path = os.path.join(DATA_DIR, "scrapy_players.json")
print(f"✓ Scrapy raw JSON: {scrapy_json_path}")

# Summary
print(f"\n{'=' * 60}")
print("FINAL OUTPUT FILES")
print(f"{'=' * 60}")
for f in sorted(os.listdir(DATA_DIR)):
    fpath = os.path.join(DATA_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f"  {f:40s} {size:>10,} bytes")

print(f"\n✓ Project complete. Final DataFrame shape: {df_players.shape}")
print(f"  Ready for analysis: correlating age, position, and market valuation.")

✓ Saved: data/players_full.csv (528 rows, 30 columns)
✓ Saved: data/market_value_history.csv (215 rows)
✓ Scrapy raw JSON: data/scrapy_players.json

FINAL OUTPUT FILES
  market_value_history.csv                     29,113 bytes
  players_full.csv                            115,020 bytes
  scrapy_players.json                         189,623 bytes

✓ Project complete. Final DataFrame shape: (528, 30)
  Ready for analysis: correlating age, position, and market valuation.


---

## Conclusion

This project demonstrates a complete web scraping pipeline using all required tools:

| Tool | Usage |
|------|-------|
| **Scrapy** | Multi-level crawl of league → clubs → squad tables with pagination, respecting robots.txt |
| **requests + BeautifulSoup** | Fetching and parsing static player profile pages with CSS selectors |
| **Selenium** | Handling cookie consent + extracting dynamically-rendered Highcharts market value data |
| **Python regex** | Parsing financial values (€45.00m → 45000000), dates, heights, and cleaning whitespace |
| **Pandas** | Structuring, merging, cleaning, and analyzing the final datasets |

The final output consists of:
- A **player-level DataFrame** with 500+ Premier League players and their attributes
- A **market value history DataFrame** tracking how top players' valuations changed over time
- Both datasets exported as CSV, ready for further statistical analysis